## Coding Applications with Gradient Descent

In [2]:
# Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm, kstest, anderson, shapiro
# for comparison only
from sklearn.linear_model import LinearRegression, Ridge

In [3]:
# get some real data to play with for gradient descent
data = pd.read_csv('https://github.com/yanhai-xiong/DATA301-AML/blob/main/Data%20Sets/Advertising.csv?raw=true')

In [4]:
# here the goal is to predict the Sales (cont. variable) by using values from TV, Radio and Newspaper advertising
x = data.loc[:,['TV','Radio','Newspaper']].values
y = data['Sales'].values

$$\large Sales \approx w_1*\text{TV} + w_2*\text{Radio} + w_3*{Newspaper} + w_0*1 $$

In [5]:
# first we want to center the data, and then we want to find the optimal weights for a linear model, i.e. minimize the sum of squared errors
# we may need a scaler
def zscore(x):
    return (x-np.mean(x,axis=0))/np.std(x,axis=0)

In [6]:
xscaled = zscore(x)
# ycentered = y-np.mean(y) # this is only to show and tell, but it is not what we prefer in practice

In [7]:
# let's repeat the gradient descent by including a bias term and by NOT centereing the values of y
x_aug = np.column_stack([np.ones(len(xscaled)),xscaled])

In [8]:
# we look for the optimal weights w1, w2 and w3 such that the predictions made with the corresponding linear combination of the columns is minimizing SSE
# define the loss function
# if we use ycentered we do not need an intercept term 
def L(w):
    prediction = x_aug@w
    errors = y - prediction
    return sum((errors**2))/len(errors) # this is MSE

In [9]:
def gradient(w):
    prediction = x_aug@w
    errors = y - prediction
    return (-1/len(errors))*errors@x_aug # here we applied the Chain Rule to differentiate; also the linear algebra behind matrix-vector products

In [11]:
w = [1,2,3,4]
L(w)

np.float64(191.95117135954678)

### We want to improve w using RIDGE


In [13]:
def L_ridge(x_aug,w):
    prediction = x_aug@w 
    errors = y - prediction
    return sum((errors**2))/len(errors) + alpha*sum(w**2)

In [12]:
def gradient_ridge(x_aug, w):
    prediction = x_aug @ w
    errors = y - prediction

    # Ridge penalty: do not regularize the bias w[0]
    penalty = 2 * alpha * np.array(w)
    penalty[0] = 0

    return (-2 / len(errors)) * errors @ x_aug + penalty

In [24]:
# apply GD for training the Ridge coefficients
# we want the "optimal" value for the weights
maxiter = 2000
learning_rate = 0.01
# we want some stopping criteria, for example if the weights are no longer significantly updated
# we measure the difference between w_new and w_old vs epsilon -> tolerance for deciding convergence.
tol = 1e-6
w_old = [0,1,2,3]
alpha = 0.01

In [25]:
# how to implement the gradient descent
for it in range(maxiter):
    w_new = w_old - learning_rate*gradient_ridge(x_aug, w_old)

    if max(abs(w_new-w_old))<tol:
        print("The algorithm has converged!")
        break

    w_old = w_new
    if (it+1)%100==0:
        print('Iteration', it+1, 'The MSE is : '+str(L(w_old)))

Iteration 100 The MSE is : 6.808509164759016
Iteration 200 The MSE is : 2.885177522508095
Iteration 300 The MSE is : 2.7917537444236147
Iteration 400 The MSE is : 2.7871757537964448
Iteration 500 The MSE is : 2.786591732540893
Iteration 600 The MSE is : 2.7864702458392903
Iteration 700 The MSE is : 2.7864405074491128
The algorithm has converged!


In [26]:
w_new

array([ 1.40224974e+01,  3.88135838e+00,  2.76187559e+00, -9.55787691e-03])

### Let's compare with Sklearn's built-in functions

In [27]:
# fit_intercept=False because we already manually added one for intercept
model = LinearRegression(fit_intercept=False)
model.fit(x_aug, y)
model.coef_

array([14.0225    ,  3.91925365,  2.79206274, -0.02253861])

In [31]:
from sklearn.linear_model import Ridge
# Note that sklearn's RIDGE doesn't /len(errors) for loss and regularization 

model = Ridge(alpha=0.01, fit_intercept=True)
model.fit(xscaled, y)
print("Bias:", model.intercept_)
print("Weights:", model.coef_)

Bias: 14.0225
Weights: [ 3.91906232  2.79191028 -0.02247266]


## Adaptive Momentum Gradient Descent Adam

In [ ]:
def gradient(w,ind):
    prediction = x_aug[ind]@w
    errors = y[ind] - prediction
    return (-2/len(errors))*errors@x_aug[ind]

In [54]:
# we want the "optimal" value for the weights
t = 0
done = False

epochs = 10000
num_batch = 12
learning_rate = 0.01

# Adam parameters
beta1 = 0.9
beta2 = 0.999
epsilon = 1e-8

tol = 1e-6

# initial weights
w_old = np.array([0., 1., 2., 3.])

# initialize Adam's first and second moments
m = np.zeros(len(w_old))
v = np.zeros(len(w_old))

In [56]:
# how to implement the mini-batch adaptive gradient descent
for epoch in range(epochs):
    ind = np.random.permutation(range(len(x)))
    batches = np.array_split(ind,len(x)//num_batch)
    for ind in batches:
        g = gradient(w_old,ind)
        m = beta1*m + (1-beta1)*g
        v = beta2*v +(1-beta2)*(g**2)
        m_hat = m/(1-beta1**(t+1))
        v_hat = v/(1-beta2**(t+1))
        w_new = w_old - learning_rate*m_hat/(np.sqrt(v_hat)+epsilon)
        
        if max(abs(w_new-w_old))<tol:
            done = True
            break
        t += 1
        w_old = w_new
    if (epoch+1)%1000==0:
        print('The MSE is : '+str(L(w_old)))
    if done:
        print("The algorithm has converged!")
        break

The MSE is : 2.784804217178561
The MSE is : 2.784385810993722
The MSE is : 2.7849407606046284
The MSE is : 2.7847376294501047
The MSE is : 2.7847151997302677
The MSE is : 2.7847099526280177
The MSE is : 2.784328410031764
The MSE is : 2.7843750063796064
The MSE is : 2.7842200797186547
The MSE is : 2.784223830515488
